# Exploratory Data Analysis (EDA) - Datasets de Audio

Este notebook analiza las estadísticas clave de los dos datasets de audio utilizados en el proyecto:

1. **NeuroVoz** - Dataset español de vocales sostenidas (HC vs PD)
2. **PC-GITA** - Dataset colombiano de vocales sostenidas (Control vs Patológicas)

Se analizan:
- Número de pacientes sanos vs enfermos
- Número de audios por grupo
- Distribución por sexo
- Distribución de edades
- Variables clínicas (UPDRS, H-Y, tiempo de enfermedad)
- Distribución por vocal y repetición
- Comparativa entre ambos datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Paleta de colores personalizada
COLORS = {
    'healthy': '#2ecc71',
    'pathological': '#e74c3c',
    'male': '#3498db',
    'female': '#e91e63',
    'neurovoz': '#9b59b6',
    'pcgita': '#f39c12',
}

# Rutas base
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

NEUROVOZ_DIR = PROJECT_ROOT / 'data' / 'original' / 'neurovoz'
PCGITA_DIR = PROJECT_ROOT / 'data' / 'original' / 'pc-gita'

print(f'Project root: {PROJECT_ROOT}')
print(f'NeuroVoz dir: {NEUROVOZ_DIR}')
print(f'PC-GITA dir:  {PCGITA_DIR}')

---
## 1. Carga de Datos

In [ ]:
# ============================================================
# NEUROVOZ - Cargar metadata filtrada (solo vocales)
# ============================================================
nv_hc = pd.read_csv(NEUROVOZ_DIR / 'metadata' / 'metadata_hc_vocales.csv')
nv_pd = pd.read_csv(NEUROVOZ_DIR / 'metadata' / 'metadata_pd_vocales.csv')

# Añadir columna de grupo unificada
nv_hc['Label'] = 'HC'
nv_pd['Label'] = 'PD'

# Unir ambos
nv = pd.concat([nv_hc, nv_pd], ignore_index=True)

# Mapear sexo: 1 = Hombre, 0 = Mujer (según convención del dataset)
nv['Sex_label'] = nv['Sex'].map({1.0: 'Hombre', 0.0: 'Mujer'})

# Extraer vocal y repetición del path del audio
# Formato: data/audios_reorganizados/Control/A/0034_A1.wav
nv['Filename'] = nv['Audio'].apply(lambda x: str(x).split('/')[-1] if pd.notna(x) else None)
nv['Vowel'] = nv['Audio'].apply(lambda x: str(x).split('/')[-2] if pd.notna(x) else None)

print(f'NeuroVoz - Total filas (audios): {len(nv)}')
print(f'  HC: {len(nv_hc)} filas')
print(f'  PD: {len(nv_pd)} filas')
print(f'\nColumnas: {list(nv.columns)}')

In [ ]:
# ============================================================
# PC-GITA - Cargar metadata
# ============================================================
gita_meta = pd.read_excel(
    PCGITA_DIR / 'metadata' / 'Copia de PCGITA_metadata.xlsx',
    sheet_name='PD+HC'
)

# Distinguir PD de HC por el nombre: AVPEPUDEAC = Control, AVPEPUDEA0 = PD
gita_meta['Label'] = gita_meta['RECODING ORIGINAL NAME'].apply(
    lambda x: 'HC' if 'AVPEPUDEAC' in str(x) else 'PD'
)
gita_meta['Sex_label'] = gita_meta['SEX'].map({'M': 'Hombre', 'F': 'Mujer'})

# Contar audios reales desde las carpetas
gita_audio_counts = {}
for group in ['Control', 'Patologicas']:
    for vowel in ['A', 'E', 'I', 'O', 'U']:
        path = PCGITA_DIR / 'audios' / group / vowel
        if path.exists():
            count = len([f for f in path.iterdir() if f.suffix == '.wav'])
            gita_audio_counts[(group, vowel)] = count

print(f'PC-GITA - Pacientes en metadata: {len(gita_meta)}')
print(f'  HC: {(gita_meta["Label"]=="HC").sum()} pacientes')
print(f'  PD: {(gita_meta["Label"]=="PD").sum()} pacientes')
print(f'\nTotal audios en carpetas: {sum(gita_audio_counts.values())}')
print(f'\nColumnas: {list(gita_meta.columns)}')

---
## 2. NeuroVoz - Análisis Detallado

### 2.1 Pacientes y Audios

In [ ]:
# ============================================================
# Estadísticas de pacientes y audios
# ============================================================
nv_patients = nv.groupby('Label')['ID'].nunique()
nv_audios = nv.groupby('Label').size()

print('=' * 50)
print('NEUROVOZ - RESUMEN DE PACIENTES Y AUDIOS')
print('=' * 50)
print(f'\nPacientes sanos (HC):    {nv_patients.get("HC", 0)}')
print(f'Pacientes enfermos (PD): {nv_patients.get("PD", 0)}')
print(f'Total pacientes:         {nv["ID"].nunique()}')
print(f'\nAudios sanos (HC):       {nv_audios.get("HC", 0)}')
print(f'Audios enfermos (PD):    {nv_audios.get("PD", 0)}')
print(f'Total audios:            {len(nv)}')
print(f'\nMedia audios/paciente HC: {nv_audios.get("HC",0)/nv_patients.get("HC",1):.1f}')
print(f'Media audios/paciente PD: {nv_audios.get("PD",0)/nv_patients.get("PD",1):.1f}')

In [ ]:
# ============================================================
# Gráfico: Pacientes y Audios por grupo
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pacientes por grupo
bars1 = axes[0].bar(
    ['HC (Sanos)', 'PD (Enfermos)'],
    [nv_patients.get('HC', 0), nv_patients.get('PD', 0)],
    color=[COLORS['healthy'], COLORS['pathological']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[0].set_title('NeuroVoz - Pacientes por Grupo', fontweight='bold')
axes[0].set_ylabel('Número de pacientes')
axes[0].set_ylim(0, max(nv_patients) * 1.2)

# Audios por grupo
bars2 = axes[1].bar(
    ['HC (Sanos)', 'PD (Enfermos)'],
    [nv_audios.get('HC', 0), nv_audios.get('PD', 0)],
    color=[COLORS['healthy'], COLORS['pathological']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[1].set_title('NeuroVoz - Audios por Grupo', fontweight='bold')
axes[1].set_ylabel('Número de audios')
axes[1].set_ylim(0, max(nv_audios) * 1.2)

plt.tight_layout()
plt.show()

### 2.2 Distribución por Sexo

In [ ]:
# ============================================================
# Sexo por grupo (a nivel de paciente único)
# ============================================================
nv_sex = nv.drop_duplicates(subset=['ID', 'Label']).groupby(['Label', 'Sex_label']).size().unstack(fill_value=0)

print('NEUROVOZ - Distribución por Sexo (pacientes únicos)')
print('-' * 50)
print(nv_sex)
print()

# Porcentajes
for label in ['HC', 'PD']:
    if label in nv_sex.index:
        total = nv_sex.loc[label].sum()
        for sex in nv_sex.columns:
            pct = nv_sex.loc[label, sex] / total * 100
            print(f'  {label} - {sex}: {nv_sex.loc[label, sex]} ({pct:.1f}%)')

In [ ]:
# ============================================================
# Gráfico: Distribución por Sexo
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, label in enumerate(['HC', 'PD']):
    if label in nv_sex.index:
        values = nv_sex.loc[label]
        colors_pie = [COLORS['male'] if 'Hombre' in s else COLORS['female'] for s in values.index]
        wedges, texts, autotexts = axes[idx].pie(
            values, labels=values.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            textprops={'fontsize': 12}, pctdistance=0.7,
            wedgeprops=dict(edgecolor='white', linewidth=2)
        )
        for autotext in autotexts:
            autotext.set_fontweight('bold')
        title = 'Sanos (HC)' if label == 'HC' else 'Enfermos (PD)'
        total = values.sum()
        axes[idx].set_title(f'NeuroVoz - {title}\n(n={total} pacientes)', fontweight='bold')

plt.suptitle('Distribución por Sexo - NeuroVoz', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Distribución de Edades

In [ ]:
# ============================================================
# Edad por grupo (pacientes únicos)
# ============================================================
nv_unique = nv.drop_duplicates(subset=['ID', 'Label'])

print('NEUROVOZ - Estadísticas de Edad')
print('-' * 50)
for label in ['HC', 'PD']:
    ages = nv_unique[nv_unique['Label'] == label]['Age'].dropna()
    print(f'\n  {label}:')
    print(f'    Media:   {ages.mean():.1f} años')
    print(f'    Mediana: {ages.median():.1f} años')
    print(f'    Std:     {ages.std():.1f} años')
    print(f'    Rango:   {ages.min():.0f} - {ages.max():.0f} años')

In [ ]:
# ============================================================
# Gráfico: Distribución de edades
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
for label, color, name in [('HC', COLORS['healthy'], 'Sanos'), ('PD', COLORS['pathological'], 'Enfermos')]:
    ages = nv_unique[nv_unique['Label'] == label]['Age'].dropna()
    axes[0].hist(ages, bins=15, alpha=0.6, label=f'{name} (n={len(ages)})', color=color, edgecolor='white')
axes[0].set_title('NeuroVoz - Distribución de Edades', fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot
data_box = [nv_unique[nv_unique['Label'] == 'HC']['Age'].dropna(),
            nv_unique[nv_unique['Label'] == 'PD']['Age'].dropna()]
bp = axes[1].boxplot(data_box, labels=['HC (Sanos)', 'PD (Enfermos)'],
                     patch_artist=True, widths=0.4,
                     medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor(COLORS['healthy'])
bp['boxes'][1].set_facecolor(COLORS['pathological'])
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_title('NeuroVoz - Boxplot de Edades', fontweight='bold')
axes[1].set_ylabel('Edad (años)')

plt.tight_layout()
plt.show()

### 2.4 Audios por Vocal

In [ ]:
# ============================================================
# Distribución de audios por vocal y grupo
# ============================================================
nv_vowel_group = nv.groupby(['Label', 'Vowel']).size().unstack(fill_value=0)

print('NEUROVOZ - Audios por Vocal y Grupo')
print('-' * 50)
print(nv_vowel_group)
print(f'\nTotal: {nv_vowel_group.sum().sum()}')

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(nv_vowel_group.columns))
width = 0.35

if 'HC' in nv_vowel_group.index:
    ax.bar(x - width/2, nv_vowel_group.loc['HC'], width, label='HC (Sanos)',
           color=COLORS['healthy'], edgecolor='white', linewidth=1.5)
if 'PD' in nv_vowel_group.index:
    ax.bar(x + width/2, nv_vowel_group.loc['PD'], width, label='PD (Enfermos)',
           color=COLORS['pathological'], edgecolor='white', linewidth=1.5)

# Añadir valores encima de las barras
for i, vowel in enumerate(nv_vowel_group.columns):
    if 'HC' in nv_vowel_group.index:
        ax.text(i - width/2, nv_vowel_group.loc['HC', vowel] + 1,
                str(nv_vowel_group.loc['HC', vowel]), ha='center', fontsize=10, fontweight='bold')
    if 'PD' in nv_vowel_group.index:
        ax.text(i + width/2, nv_vowel_group.loc['PD', vowel] + 1,
                str(nv_vowel_group.loc['PD', vowel]), ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Vocal')
ax.set_ylabel('Número de audios')
ax.set_title('NeuroVoz - Audios por Vocal y Grupo', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(nv_vowel_group.columns)
ax.legend()
ax.set_ylim(0, nv_vowel_group.max().max() * 1.2)

plt.tight_layout()
plt.show()

### 2.5 Variables Clínicas (solo PD)

In [ ]:
# ============================================================
# Variables clínicas de pacientes PD (únicos)
# ============================================================
nv_pd_unique = nv[nv['Label'] == 'PD'].drop_duplicates(subset='ID')

print('NEUROVOZ - Variables Clínicas (PD)')
print('=' * 50)

# UPDRS
updrs = nv_pd_unique['UPDRS scale'].dropna()
print(f'\nUPDRS Scale:')
print(f'  n:       {len(updrs)}')
print(f'  Media:   {updrs.mean():.1f}')
print(f'  Mediana: {updrs.median():.1f}')
print(f'  Std:     {updrs.std():.1f}')
print(f'  Rango:   {updrs.min():.0f} - {updrs.max():.0f}')

# Hoehn & Yahr
hy = nv_pd_unique['H-Y Stadium'].dropna()
print(f'\nH-Y Stadium:')
print(f'  n:       {len(hy)}')
print(f'  Distribución:')
for val, count in hy.value_counts().sort_index().items():
    print(f'    Estadio {val:.0f}: {count} pacientes ({count/len(hy)*100:.1f}%)')

# Tiempo de enfermedad
time_d = nv_pd_unique['Time Disease (years)'].dropna()
print(f'\nTiempo de enfermedad (años):')
print(f'  n:       {len(time_d)}')
print(f'  Media:   {time_d.mean():.1f}')
print(f'  Mediana: {time_d.median():.1f}')
print(f'  Rango:   {time_d.min():.0f} - {time_d.max():.0f}')

In [ ]:
# ============================================================
# Gráficos: Variables Clínicas NeuroVoz PD
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# UPDRS
updrs = nv_pd_unique['UPDRS scale'].dropna()
axes[0].hist(updrs, bins=12, color=COLORS['pathological'], alpha=0.7, edgecolor='white')
axes[0].axvline(updrs.mean(), color='black', linestyle='--', linewidth=2, label=f'Media={updrs.mean():.1f}')
axes[0].set_title('UPDRS Scale', fontweight='bold')
axes[0].set_xlabel('UPDRS')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# H-Y Stadium
hy = nv_pd_unique['H-Y Stadium'].dropna()
hy_counts = hy.value_counts().sort_index()
axes[1].bar(hy_counts.index.astype(str), hy_counts.values,
            color=COLORS['pathological'], alpha=0.7, edgecolor='white', linewidth=1.5)
for i, (val, count) in enumerate(hy_counts.items()):
    axes[1].text(i, count + 0.3, str(count), ha='center', fontweight='bold')
axes[1].set_title('Hoehn & Yahr Stadium', fontweight='bold')
axes[1].set_xlabel('Estadio')
axes[1].set_ylabel('Frecuencia')

# Tiempo de enfermedad
time_d = nv_pd_unique['Time Disease (years)'].dropna()
axes[2].hist(time_d, bins=12, color=COLORS['pathological'], alpha=0.7, edgecolor='white')
axes[2].axvline(time_d.mean(), color='black', linestyle='--', linewidth=2, label=f'Media={time_d.mean():.1f}')
axes[2].set_title('Tiempo de Enfermedad', fontweight='bold')
axes[2].set_xlabel('Años')
axes[2].set_ylabel('Frecuencia')
axes[2].legend()

plt.suptitle('NeuroVoz - Variables Clínicas (grupo PD)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.6 Síntomas clínicos (PD)

In [ ]:
# ============================================================
# Prevalencia de síntomas en pacientes PD
# ============================================================
symptom_cols = ['Vocal tremor', 'Cephalic tremor', 'Mandibular tremor',
                'Sialorrhoea', 'Dysphagia', 'Hypophonic voice']

symptom_data = nv_pd_unique[symptom_cols]
symptom_pct = (symptom_data.sum() / len(symptom_data) * 100).sort_values(ascending=True)

print('NEUROVOZ - Prevalencia de Síntomas en PD')
print('-' * 50)
for sym, pct in symptom_pct.items():
    print(f'  {sym:25s}: {pct:5.1f}% ({int(symptom_data[sym].sum())}/{len(symptom_data)})')

# Gráfico horizontal
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(symptom_pct)), symptom_pct.values,
               color=COLORS['pathological'], alpha=0.7, edgecolor='white', linewidth=1.5)
ax.set_yticks(range(len(symptom_pct)))
ax.set_yticklabels(symptom_pct.index)
ax.set_xlabel('Prevalencia (%)')
ax.set_title('NeuroVoz - Prevalencia de Síntomas en PD', fontweight='bold')
ax.set_xlim(0, 100)

for i, (val, name) in enumerate(zip(symptom_pct.values, symptom_pct.index)):
    ax.text(val + 1, i, f'{val:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. PC-GITA - Análisis Detallado

### 3.1 Pacientes y Audios

In [ ]:
# ============================================================
# Estadísticas de pacientes y audios
# ============================================================
gita_patients = gita_meta.groupby('Label').size()

# Audios totales contados de las carpetas
gita_hc_audios = sum(v for (g, _), v in gita_audio_counts.items() if g == 'Control')
gita_pd_audios = sum(v for (g, _), v in gita_audio_counts.items() if g == 'Patologicas')

print('=' * 50)
print('PC-GITA - RESUMEN DE PACIENTES Y AUDIOS')
print('=' * 50)
print(f'\nPacientes sanos (HC):    {gita_patients.get("HC", 0)}')
print(f'Pacientes enfermos (PD): {gita_patients.get("PD", 0)}')
print(f'Total pacientes:         {len(gita_meta)}')
print(f'\nAudios sanos (HC):       {gita_hc_audios}')
print(f'Audios enfermos (PD):    {gita_pd_audios}')
print(f'Total audios:            {gita_hc_audios + gita_pd_audios}')
print(f'\nMedia audios/paciente HC: {gita_hc_audios/gita_patients.get("HC",1):.1f}')
print(f'Media audios/paciente PD: {gita_pd_audios/gita_patients.get("PD",1):.1f}')
print(f'\n(5 vocales x 3 repeticiones = 15 audios/paciente esperados)')

In [ ]:
# ============================================================
# Gráfico: Pacientes y Audios por grupo
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pacientes
bars1 = axes[0].bar(
    ['HC (Sanos)', 'PD (Enfermos)'],
    [gita_patients.get('HC', 0), gita_patients.get('PD', 0)],
    color=[COLORS['healthy'], COLORS['pathological']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[0].set_title('PC-GITA - Pacientes por Grupo', fontweight='bold')
axes[0].set_ylabel('Número de pacientes')
axes[0].set_ylim(0, max(gita_patients) * 1.3)

# Audios
bars2 = axes[1].bar(
    ['HC (Sanos)', 'PD (Enfermos)'],
    [gita_hc_audios, gita_pd_audios],
    color=[COLORS['healthy'], COLORS['pathological']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold', fontsize=14)
axes[1].set_title('PC-GITA - Audios por Grupo', fontweight='bold')
axes[1].set_ylabel('Número de audios')
axes[1].set_ylim(0, max(gita_hc_audios, gita_pd_audios) * 1.2)

plt.tight_layout()
plt.show()

### 3.2 Distribución por Sexo

In [ ]:
# ============================================================
# Sexo por grupo
# ============================================================
gita_sex = gita_meta.groupby(['Label', 'Sex_label']).size().unstack(fill_value=0)

print('PC-GITA - Distribución por Sexo')
print('-' * 50)
print(gita_sex)
print()

for label in ['HC', 'PD']:
    if label in gita_sex.index:
        total = gita_sex.loc[label].sum()
        for sex in gita_sex.columns:
            pct = gita_sex.loc[label, sex] / total * 100
            print(f'  {label} - {sex}: {gita_sex.loc[label, sex]} ({pct:.1f}%)')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, label in enumerate(['HC', 'PD']):
    if label in gita_sex.index:
        values = gita_sex.loc[label]
        colors_pie = [COLORS['male'] if 'Hombre' in s else COLORS['female'] for s in values.index]
        wedges, texts, autotexts = axes[idx].pie(
            values, labels=values.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            textprops={'fontsize': 12}, pctdistance=0.7,
            wedgeprops=dict(edgecolor='white', linewidth=2)
        )
        for autotext in autotexts:
            autotext.set_fontweight('bold')
        title = 'Sanos (HC)' if label == 'HC' else 'Enfermos (PD)'
        total = values.sum()
        axes[idx].set_title(f'PC-GITA - {title}\n(n={total} pacientes)', fontweight='bold')

plt.suptitle('Distribución por Sexo - PC-GITA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Distribución de Edades

In [ ]:
# ============================================================
# Edad por grupo
# ============================================================
print('PC-GITA - Estadísticas de Edad')
print('-' * 50)
for label in ['HC', 'PD']:
    ages = gita_meta[gita_meta['Label'] == label]['AGE'].dropna()
    print(f'\n  {label}:')
    print(f'    Media:   {ages.mean():.1f} años')
    print(f'    Mediana: {ages.median():.1f} años')
    print(f'    Std:     {ages.std():.1f} años')
    print(f'    Rango:   {ages.min():.0f} - {ages.max():.0f} años')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [('HC', COLORS['healthy'], 'Sanos'), ('PD', COLORS['pathological'], 'Enfermos')]:
    ages = gita_meta[gita_meta['Label'] == label]['AGE'].dropna()
    axes[0].hist(ages, bins=15, alpha=0.6, label=f'{name} (n={len(ages)})', color=color, edgecolor='white')
axes[0].set_title('PC-GITA - Distribución de Edades', fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

data_box = [gita_meta[gita_meta['Label'] == 'HC']['AGE'].dropna(),
            gita_meta[gita_meta['Label'] == 'PD']['AGE'].dropna()]
bp = axes[1].boxplot(data_box, labels=['HC (Sanos)', 'PD (Enfermos)'],
                     patch_artist=True, widths=0.4,
                     medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor(COLORS['healthy'])
bp['boxes'][1].set_facecolor(COLORS['pathological'])
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_title('PC-GITA - Boxplot de Edades', fontweight='bold')
axes[1].set_ylabel('Edad (años)')

plt.tight_layout()
plt.show()

### 3.4 Audios por Vocal

In [ ]:
# ============================================================
# Distribución de audios por vocal y grupo
# ============================================================
vowels = ['A', 'E', 'I', 'O', 'U']
gita_hc_by_vowel = [gita_audio_counts.get(('Control', v), 0) for v in vowels]
gita_pd_by_vowel = [gita_audio_counts.get(('Patologicas', v), 0) for v in vowels]

print('PC-GITA - Audios por Vocal y Grupo')
print('-' * 50)
print(f'{"Vocal":>8} {"HC":>8} {"PD":>8} {"Total":>8}')
for i, v in enumerate(vowels):
    print(f'{v:>8} {gita_hc_by_vowel[i]:>8} {gita_pd_by_vowel[i]:>8} {gita_hc_by_vowel[i]+gita_pd_by_vowel[i]:>8}')
print(f'{"TOTAL":>8} {sum(gita_hc_by_vowel):>8} {sum(gita_pd_by_vowel):>8} {sum(gita_hc_by_vowel)+sum(gita_pd_by_vowel):>8}')

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(vowels))
width = 0.35

ax.bar(x - width/2, gita_hc_by_vowel, width, label='HC (Sanos)',
       color=COLORS['healthy'], edgecolor='white', linewidth=1.5)
ax.bar(x + width/2, gita_pd_by_vowel, width, label='PD (Enfermos)',
       color=COLORS['pathological'], edgecolor='white', linewidth=1.5)

for i, v in enumerate(vowels):
    ax.text(i - width/2, gita_hc_by_vowel[i] + 1, str(gita_hc_by_vowel[i]),
            ha='center', fontsize=10, fontweight='bold')
    ax.text(i + width/2, gita_pd_by_vowel[i] + 1, str(gita_pd_by_vowel[i]),
            ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Vocal')
ax.set_ylabel('Número de audios')
ax.set_title('PC-GITA - Audios por Vocal y Grupo', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(vowels)
ax.legend()
ax.set_ylim(0, max(max(gita_hc_by_vowel), max(gita_pd_by_vowel)) * 1.2)

plt.tight_layout()
plt.show()

### 3.5 Variables Clínicas (solo PD)

In [ ]:
# ============================================================
# Variables clínicas de pacientes PD en PC-GITA
# ============================================================
gita_pd = gita_meta[gita_meta['Label'] == 'PD']

print('PC-GITA - Variables Clínicas (PD)')
print('=' * 50)

# UPDRS
updrs_g = gita_pd['UPDRS'].dropna()
print(f'\nUPDRS:')
print(f'  n:       {len(updrs_g)}')
print(f'  Media:   {updrs_g.mean():.1f}')
print(f'  Mediana: {updrs_g.median():.1f}')
print(f'  Std:     {updrs_g.std():.1f}')
print(f'  Rango:   {updrs_g.min():.0f} - {updrs_g.max():.0f}')

# UPDRS-speech
updrs_speech = gita_pd['UPDRS-speech'].dropna()
print(f'\nUPDRS-Speech:')
print(f'  n:       {len(updrs_speech)}')
print(f'  Distribución:')
for val, count in updrs_speech.value_counts().sort_index().items():
    print(f'    Nivel {val:.0f}: {count} pacientes ({count/len(updrs_speech)*100:.1f}%)')

# H&Y
hy_g = gita_pd['H/Y'].dropna()
print(f'\nH-Y Stadium:')
print(f'  n:       {len(hy_g)}')
print(f'  Distribución:')
for val, count in hy_g.value_counts().sort_index().items():
    print(f'    Estadio {val:.0f}: {count} pacientes ({count/len(hy_g)*100:.1f}%)')

# Tiempo tras diagnóstico
time_g = gita_pd['time after diagnosis'].dropna()
print(f'\nTiempo tras diagnóstico (años):')
print(f'  n:       {len(time_g)}')
print(f'  Media:   {time_g.mean():.1f}')
print(f'  Mediana: {time_g.median():.1f}')
print(f'  Rango:   {time_g.min():.0f} - {time_g.max():.0f}')

In [ ]:
# ============================================================
# Gráficos: Variables Clínicas PC-GITA PD
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# UPDRS
updrs_g = gita_pd['UPDRS'].dropna()
axes[0].hist(updrs_g, bins=12, color=COLORS['pathological'], alpha=0.7, edgecolor='white')
axes[0].axvline(updrs_g.mean(), color='black', linestyle='--', linewidth=2, label=f'Media={updrs_g.mean():.1f}')
axes[0].set_title('UPDRS', fontweight='bold')
axes[0].set_xlabel('UPDRS')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# H-Y
hy_g = gita_pd['H/Y'].dropna()
hy_counts_g = hy_g.value_counts().sort_index()
axes[1].bar(hy_counts_g.index.astype(str), hy_counts_g.values,
            color=COLORS['pathological'], alpha=0.7, edgecolor='white', linewidth=1.5)
for i, (val, count) in enumerate(hy_counts_g.items()):
    axes[1].text(i, count + 0.3, str(count), ha='center', fontweight='bold')
axes[1].set_title('Hoehn & Yahr Stadium', fontweight='bold')
axes[1].set_xlabel('Estadio')
axes[1].set_ylabel('Frecuencia')

# Tiempo tras diagnóstico
time_g = gita_pd['time after diagnosis'].dropna()
axes[2].hist(time_g, bins=12, color=COLORS['pathological'], alpha=0.7, edgecolor='white')
axes[2].axvline(time_g.mean(), color='black', linestyle='--', linewidth=2, label=f'Media={time_g.mean():.1f}')
axes[2].set_title('Tiempo tras Diagnóstico', fontweight='bold')
axes[2].set_xlabel('Años')
axes[2].set_ylabel('Frecuencia')
axes[2].legend()

plt.suptitle('PC-GITA - Variables Clínicas (grupo PD)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Comparativa entre Datasets

In [ ]:
# ============================================================
# Tabla resumen comparativa
# ============================================================
nv_unique_all = nv.drop_duplicates(subset=['ID', 'Label'])

comparison = pd.DataFrame({
    'Métrica': [
        'País de origen',
        'Idioma',
        'Total pacientes',
        'Pacientes HC',
        'Pacientes PD',
        'Total audios (vocales)',
        'Audios HC',
        'Audios PD',
        'Vocales disponibles',
        'Repeticiones/vocal',
        'Audios/paciente (media)',
        'Edad media HC',
        'Edad media PD',
        'Rango edad HC',
        'Rango edad PD',
        '% Mujeres HC',
        '% Mujeres PD',
        'Equilibrio sexo',
    ],
    'NeuroVoz': [
        'España',
        'Español (España)',
        nv['ID'].nunique(),
        nv_patients.get('HC', 0),
        nv_patients.get('PD', 0),
        len(nv),
        nv_audios.get('HC', 0),
        nv_audios.get('PD', 0),
        ', '.join(sorted(nv['Vowel'].dropna().unique())),
        'Variable (1-3)',
        f"{len(nv) / nv['ID'].nunique():.1f}",
        f"{nv_unique_all[nv_unique_all['Label']=='HC']['Age'].mean():.1f}",
        f"{nv_unique_all[nv_unique_all['Label']=='PD']['Age'].mean():.1f}",
        f"{nv_unique_all[nv_unique_all['Label']=='HC']['Age'].min():.0f}-{nv_unique_all[nv_unique_all['Label']=='HC']['Age'].max():.0f}",
        f"{nv_unique_all[nv_unique_all['Label']=='PD']['Age'].min():.0f}-{nv_unique_all[nv_unique_all['Label']=='PD']['Age'].max():.0f}",
        f"{(nv_sex.loc['HC'].get('Mujer',0)/nv_sex.loc['HC'].sum()*100):.1f}%" if 'HC' in nv_sex.index else 'N/A',
        f"{(nv_sex.loc['PD'].get('Mujer',0)/nv_sex.loc['PD'].sum()*100):.1f}%" if 'PD' in nv_sex.index else 'N/A',
        'No balanceado' if 'HC' in nv_sex.index and abs(nv_sex.loc['HC'].get('Mujer',0) - nv_sex.loc['HC'].get('Hombre',0)) > 5 else 'Balanceado',
    ],
    'PC-GITA': [
        'Colombia',
        'Español (Colombia)',
        len(gita_meta),
        gita_patients.get('HC', 0),
        gita_patients.get('PD', 0),
        sum(gita_audio_counts.values()),
        gita_hc_audios,
        gita_pd_audios,
        'A, E, I, O, U',
        '3 fijas',
        f"{sum(gita_audio_counts.values()) / len(gita_meta):.1f}",
        f"{gita_meta[gita_meta['Label']=='HC']['AGE'].mean():.1f}",
        f"{gita_meta[gita_meta['Label']=='PD']['AGE'].mean():.1f}",
        f"{gita_meta[gita_meta['Label']=='HC']['AGE'].min():.0f}-{gita_meta[gita_meta['Label']=='HC']['AGE'].max():.0f}",
        f"{gita_meta[gita_meta['Label']=='PD']['AGE'].min():.0f}-{gita_meta[gita_meta['Label']=='PD']['AGE'].max():.0f}",
        f"{(gita_sex.loc['HC'].get('Mujer',0)/gita_sex.loc['HC'].sum()*100):.1f}%",
        f"{(gita_sex.loc['PD'].get('Mujer',0)/gita_sex.loc['PD'].sum()*100):.1f}%",
        'Perfectamente balanceado (25M/25F)',
    ]
})

print('COMPARATIVA ENTRE DATASETS')
print('=' * 80)
display(comparison.set_index('Métrica').style.set_properties(**{'text-align': 'center'}))

In [ ]:
# ============================================================
# Gráfico comparativo: Pacientes por dataset y grupo
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Pacientes por dataset
datasets = ['NeuroVoz', 'PC-GITA']
hc_patients = [nv_patients.get('HC', 0), gita_patients.get('HC', 0)]
pd_patients_vals = [nv_patients.get('PD', 0), gita_patients.get('PD', 0)]

x = np.arange(len(datasets))
width = 0.35
axes[0].bar(x - width/2, hc_patients, width, label='HC (Sanos)', color=COLORS['healthy'],
            edgecolor='white', linewidth=1.5)
axes[0].bar(x + width/2, pd_patients_vals, width, label='PD (Enfermos)', color=COLORS['pathological'],
            edgecolor='white', linewidth=1.5)
for i in range(len(datasets)):
    axes[0].text(i - width/2, hc_patients[i] + 0.5, str(hc_patients[i]), ha='center', fontweight='bold')
    axes[0].text(i + width/2, pd_patients_vals[i] + 0.5, str(pd_patients_vals[i]), ha='center', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(datasets)
axes[0].set_title('Pacientes por Dataset', fontweight='bold')
axes[0].set_ylabel('Número de pacientes')
axes[0].legend()

# 2. Audios por dataset
hc_audios = [nv_audios.get('HC', 0), gita_hc_audios]
pd_audios_vals = [nv_audios.get('PD', 0), gita_pd_audios]

axes[1].bar(x - width/2, hc_audios, width, label='HC (Sanos)', color=COLORS['healthy'],
            edgecolor='white', linewidth=1.5)
axes[1].bar(x + width/2, pd_audios_vals, width, label='PD (Enfermos)', color=COLORS['pathological'],
            edgecolor='white', linewidth=1.5)
for i in range(len(datasets)):
    axes[1].text(i - width/2, hc_audios[i] + 3, str(hc_audios[i]), ha='center', fontweight='bold')
    axes[1].text(i + width/2, pd_audios_vals[i] + 3, str(pd_audios_vals[i]), ha='center', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(datasets)
axes[1].set_title('Audios por Dataset', fontweight='bold')
axes[1].set_ylabel('Número de audios')
axes[1].legend()

# 3. Distribución de edades combinada
nv_ages_hc = nv_unique_all[nv_unique_all['Label']=='HC']['Age'].dropna()
nv_ages_pd = nv_unique_all[nv_unique_all['Label']=='PD']['Age'].dropna()
gita_ages_hc = gita_meta[gita_meta['Label']=='HC']['AGE'].dropna()
gita_ages_pd = gita_meta[gita_meta['Label']=='PD']['AGE'].dropna()

data_ages = [nv_ages_hc, nv_ages_pd, gita_ages_hc, gita_ages_pd]
labels_ages = ['NV-HC', 'NV-PD', 'GITA-HC', 'GITA-PD']
colors_box = [COLORS['healthy'], COLORS['pathological'], COLORS['healthy'], COLORS['pathological']]

bp = axes[2].boxplot(data_ages, labels=labels_ages, patch_artist=True, widths=0.5,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Comparativa de Edades', fontweight='bold')
axes[2].set_ylabel('Edad (años)')

plt.suptitle('Comparativa NeuroVoz vs PC-GITA', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Resumen y Conclusiones

In [ ]:
# ============================================================
# Resumen final
# ============================================================
total_patients = nv['ID'].nunique() + len(gita_meta)
total_audios = len(nv) + sum(gita_audio_counts.values())

print('=' * 60)
print('RESUMEN GLOBAL DEL PROYECTO')
print('=' * 60)
print(f'\n  Total pacientes (ambos datasets): {total_patients}')
print(f'  Total audios (ambos datasets):    {total_audios}')
print()
print('  NEUROVOZ:')
print(f'    - {nv_patients.get("HC",0)} pacientes HC + {nv_patients.get("PD",0)} pacientes PD')
print(f'    - {nv_audios.get("HC",0)} audios HC + {nv_audios.get("PD",0)} audios PD = {len(nv)} total')
print(f'    - Vocales: {sorted(nv["Vowel"].dropna().unique())}')
print()
print('  PC-GITA:')
print(f'    - {gita_patients.get("HC",0)} pacientes HC + {gita_patients.get("PD",0)} pacientes PD')
print(f'    - {gita_hc_audios} audios HC + {gita_pd_audios} audios PD = {gita_hc_audios+gita_pd_audios} total')
print(f'    - Vocales: A, E, I, O, U (5 vocales x 3 repeticiones = 15 audios/paciente)')
print()
print('  OBSERVACIONES CLAVE:')
print('    1. PC-GITA está perfectamente balanceado (50/50 PD/HC, 25M/25F por grupo)')
print('    2. NeuroVoz tiene un número variable de repeticiones por vocal')
print('    3. Ambos datasets tienen rangos de edad similares (~60 años de media)')
print('    4. Los datasets provienen de diferentes variantes del español')
print('       (España vs Colombia), lo que añade diversidad lingüística')